c:\Users\Siddharth\AppData\Local\Programs\Python\Python38\lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting 2-Layer GAT training with Scheduler...
Epoch: 010, Loss: 1.3261, Train Acc: 0.5431, Val Acc: 0.5131
Epoch: 020, Loss: 1.2857, Train Acc: 0.5630, Val Acc: 0.5281
Epoch: 030, Loss: 1.2617, Train Acc: 0.5505, Val Acc: 0.5091
Epoch: 040, Loss: 1.1990, Train Acc: 0.5872, Val Acc: 0.5283
Epoch: 050, Loss: 1.1549, Train Acc: 0.6197, Val Acc: 0.5300
Epoch: 060, Loss: 1.1112, Train Acc: 0.6486, Val Acc: 0.5312
Epoch: 070, Loss: 1.0552, Train Acc: 0.6765, Val Acc: 0.5295
Epoch: 080, Loss: 1.0161, Train Acc: 0.6814, Val Acc: 0.5300
Epoch: 090, Loss: 0.9934, Train Acc: 0.6983, Val Acc: 0.5291
Epoch: 100, Loss: 0.9784, Train Acc: 0.7009, Val Acc: 0.5280
Epoch: 110, Loss: 0.9711, Train Acc: 0.7063, Val Acc: 0.5266
Epoch: 120, Loss: 0.9673, Train Acc: 0.7070, Val Acc: 0.5241
Epoch: 130, Loss: 0.9656, Train Acc: 0.7079, Val Acc: 0.5244
Epoch: 140, Loss: 0.9662, Train Acc: 0.7093, Val Acc: 0.5259
Epoch: 150, Loss: 0.9665, Train Acc: 0.7081, Val Acc: 0.5246
Epoch: 160, Loss: 0.9632, Train Acc: 

## GAT v1 run

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATConv
from torch.nn import BatchNorm1d

# 1. Load Dataset and Masks
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# 2. Setup NeighborLoader for Mini-Batching
# We sample fewer neighbors here than SAGE because GAT attention is computationally heavier
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10], 
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)

# 3. Define the GAT Architecture
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super().__init__()
        # Layer 1: Multi-head attention. 
        # If hidden_channels is 64 and heads is 4, the output is 64 * 4 = 256 dimensions.
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.bn1 = BatchNorm1d(hidden_channels * heads)
        
        # Layer 2: Output layer. concat=False averages the heads instead of stacking them.
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=heads, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        # ELU (Exponential Linear Unit) is standard for GATs to handle negative values in attention
        x = F.elu(x) 
        x = F.dropout(x, p=0.6, training=self.training)
        
        x = self.conv2(x, edge_index)
        return x

# 4. Initialize Device, Model, and Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# We pass 64 as hidden_channels. With 4 heads, the actual hidden dimension becomes 256.
model = GAT(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes, heads=8).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, verbose=True)

# 5. Training and Testing Functions
def train():
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index)
        
        # Calculate loss only for the target nodes in the batch
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size])
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
        
    return total_loss / int(data.train_mask.sum())

@torch.no_grad()
def test():
    model.eval()
    # Full-batch inference for accurate evaluation
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = (pred[mask] == data.y[mask]).sum().item()
        acc = correct / mask.sum().item()
        accs.append(acc)
    return accs

# 6. Execution Loop
print("Starting 2-Layer GAT v2 training with Scheduler...")
for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    
    # Step the scheduler based on validation accuracy
    scheduler.step(val_acc) 
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

print(f'\nTraining Complete!')
print(f'Final Test Accuracy: {test_acc:.4f}')

## GAT v2 Baseline

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATConv
from torch_geometric.nn import GATv2Conv
from torch.nn import BatchNorm1d

# 1. Load Dataset and Masks
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# 2. Setup NeighborLoader for Mini-Batching
# We sample fewer neighbors here than SAGE because GAT attention is computationally heavier
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10], 
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)

# 3. Define the GAT Architecture
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super().__init__()
        # Layer 1: Multi-head attention. 
        # If hidden_channels is 64 and heads is 4, the output is 64 * 4 = 256 dimensions.
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads)
        self.bn1 = BatchNorm1d(hidden_channels * heads)
        
        # Layer 2: Output layer. concat=False averages the heads instead of stacking them.
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=heads, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        # ELU (Exponential Linear Unit) is standard for GATs to handle negative values in attention
        x = F.elu(x) 
        x = F.dropout(x, p=0.6, training=self.training)
        
        x = self.conv2(x, edge_index)
        return x

# 4. Initialize Device, Model, and Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# We pass 64 as hidden_channels. With 4 heads, the actual hidden dimension becomes 256.
model = GAT(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes, heads=8).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, verbose=True)

# 5. Training and Testing Functions
def train():
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index)
        
        # Calculate loss only for the target nodes in the batch
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size])
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
        
    return total_loss / int(data.train_mask.sum())

@torch.no_grad()
def test():
    model.eval()
    # Full-batch inference for accurate evaluation
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = (pred[mask] == data.y[mask]).sum().item()
        acc = correct / mask.sum().item()
        accs.append(acc)
    return accs

# 6. Execution Loop
print("Starting 2-Layer GAT v2 training with Scheduler...")
for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    
    # Step the scheduler based on validation accuracy
    scheduler.step(val_acc) 
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

print(f'\nTraining Complete!')
print(f'Final Test Accuracy: {test_acc:.4f}')

## GAT v2 with MLP head

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv, BatchNorm

# 1. Load Dataset and apply splits for benchmarking
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# 2. Setup NeighborLoader
# GATv2 is computationally more expensive than GCN/SAGE, so we sample carefully
train_loader = NeighborLoader(
    data,
    num_neighbors=[10, 5], 
    batch_size=512, # Smaller batch size to manage VRAM
    input_nodes=data.train_mask,
    shuffle=True
)

# 3. GATv2 Architecture with MLP Head
class GATv2WithMLP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        
        # --- GNN Encoder (Dynamic Attention) ---
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads)
        self.bn1 = BatchNorm(hidden_channels * heads)
        
        # concat=False averages the heads to keep feature size manageable for the MLP
        self.conv2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=False)
        self.bn2 = BatchNorm(hidden_channels)

        # --- MLP Task Head (Final Classification Logic) ---
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(hidden_channels, out_channels)
        )

    def forward(self, x, edge_index):
        # Encoder Phase
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x) # ELU is standard for GAT-based models
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        
        # Task Head Phase
        x = self.mlp(x)
        
        return x

# 4. Initialize on Local GPU (CUDA)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GATv2WithMLP(
    in_channels=dataset.num_node_features,
    hidden_channels=32, # 64 channels * 4 heads = 256 internal dimensions
    out_channels=dataset.num_classes,
    heads=4
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001, weight_decay=1e-3)

# Use a scheduler to automatically lower the learning rate when validation stalls
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True
)

# 5. Training and Benchmarking Functions
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        # Loss calculated on the 7 Flickr classes
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(data.train_mask.sum())

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs

# 6. Execution Loop
print(f"Benchmarking GATv2 + MLP Head on {device}...")

best_val_acc = 0.0  # Track the highest validation accuracy

for epoch in range(1, 201):
    # 1. Train the model
    loss = train()
    
    # 2. Get accuracy metrics for every epoch
    train_acc, val_acc, test_acc = test()
    
    # 3. Step the scheduler based on the latest val_acc
    # Doing this every epoch allows for more responsive learning rate tuning
    scheduler.step(val_acc)
    
    # 4. Check if this is the best version of GATv2
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save the best GATv2 state
        torch.save(model.state_dict(), 'GATv2_bestmodel.pt')
        # print(f"--- New Best GATv2 Model: {val_acc:.4f} at Epoch {epoch} ---")

    # 5. Periodic logging
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

# Final evaluation using the best saved weights
model.load_state_dict(torch.load('GATv2_bestmodel.pt'))
final_train_acc, final_val_acc, final_test_acc = test()

print(f'\nTraining Complete!')
print(f'Best Validation Accuracy: {best_val_acc:.4f}')
print(f'Final Test Accuracy (from best model): {final_test_acc:.4f}')

Benchmarking GATv2 + MLP Head on cuda...
Epoch: 010, Loss: 1.6716, Val Acc: 0.4448, Test Acc: 0.4327
Epoch: 020, Loss: 1.6255, Val Acc: 0.4622, Test Acc: 0.4500
Epoch: 030, Loss: 1.5953, Val Acc: 0.4852, Test Acc: 0.4757
Epoch: 040, Loss: 1.5691, Val Acc: 0.4912, Test Acc: 0.4829
Epoch: 050, Loss: 1.5500, Val Acc: 0.4961, Test Acc: 0.4922
Epoch: 060, Loss: 1.5352, Val Acc: 0.4989, Test Acc: 0.4940
Epoch: 070, Loss: 1.5213, Val Acc: 0.5020, Test Acc: 0.4966
Epoch: 080, Loss: 1.5102, Val Acc: 0.5035, Test Acc: 0.4962
Epoch: 090, Loss: 1.5006, Val Acc: 0.5067, Test Acc: 0.4996
Epoch: 100, Loss: 1.4910, Val Acc: 0.5099, Test Acc: 0.5017
Epoch: 110, Loss: 1.4827, Val Acc: 0.5097, Test Acc: 0.5032
Epoch: 120, Loss: 1.4779, Val Acc: 0.5107, Test Acc: 0.5038
Epoch: 130, Loss: 1.4728, Val Acc: 0.5124, Test Acc: 0.5050
Epoch: 140, Loss: 1.4668, Val Acc: 0.5125, Test Acc: 0.5056
Epoch: 150, Loss: 1.4672, Val Acc: 0.5125, Test Acc: 0.5057
Epoch: 160, Loss: 1.4666, Val Acc: 0.5119, Test Acc: 0.5056

C:\Users\Siddharth\AppData\Local\Temp\ipykernel_8532\1870633670.py:134: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('GATv2_bestmodel.pt'))


Training Complete!
Best Validation Accuracy: 0.5141
Final Test Accuracy (from best model): 0.5058


### ADDITION: GAT v1 with Residual dense head

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATConv
from torch.nn import BatchNorm1d


# ──────────────────────────────────────────────────────────────────────
# 1. Load Dataset
# ──────────────────────────────────────────────────────────────────────
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# ──────────────────────────────────────────────────────────────────────
# 2. Setup NeighborLoader
# ──────────────────────────────────────────────────────────────────────
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10],
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)


# ──────────────────────────────────────────────────────────────────────
# 3. Residual Dense Head (same architecture as GraphTransformer variant)
#
# DenseNet-style classifier: every block receives the concatenation
# of all previous block outputs + the original encoder embedding.
# ──────────────────────────────────────────────────────────────────────
class ResidualDenseHead(nn.Module):
    """
    A 3-block dense head with residual skip connections.

    Given encoder output h (dim = in_dim):
        Block 1:  d1 = ELU(BN(Linear(h)))                  -> dense_dim
        Block 2:  d2 = ELU(BN(Linear(h || d1)))             -> dense_dim
        Block 3:  d3 = ELU(BN(Linear(h || d1 || d2)))       -> dense_dim
        Output :  logits = Linear(h || d1 || d2 || d3)       -> out_dim

    '||' denotes concatenation.
    """
    def __init__(self, in_dim, dense_dim, out_dim, drop_p=0.5):
        super().__init__()

        # Dense block 1: takes encoder output only
        self.fc1 = nn.Linear(in_dim, dense_dim)
        self.bn1 = nn.BatchNorm1d(dense_dim)

        # Dense block 2: takes encoder output + block 1 output
        self.fc2 = nn.Linear(in_dim + dense_dim, dense_dim)
        self.bn2 = nn.BatchNorm1d(dense_dim)

        # Dense block 3: takes encoder output + blocks 1 & 2
        self.fc3 = nn.Linear(in_dim + dense_dim * 2, dense_dim)
        self.bn3 = nn.BatchNorm1d(dense_dim)

        # Final projection: all concatenated features -> classes
        self.out_proj = nn.Linear(in_dim + dense_dim * 3, out_dim)

        self.drop_p = drop_p

    def forward(self, h):
        # h: [N, in_dim]  (encoder output)

        d1 = F.elu(self.bn1(self.fc1(h)))
        d1 = F.dropout(d1, p=self.drop_p, training=self.training)

        cat1 = torch.cat([h, d1], dim=-1)                    # [N, in_dim + dense_dim]
        d2 = F.elu(self.bn2(self.fc2(cat1)))
        d2 = F.dropout(d2, p=self.drop_p, training=self.training)

        cat2 = torch.cat([h, d1, d2], dim=-1)                # [N, in_dim + 2*dense_dim]
        d3 = F.elu(self.bn3(self.fc3(cat2)))
        d3 = F.dropout(d3, p=self.drop_p, training=self.training)

        cat3 = torch.cat([h, d1, d2, d3], dim=-1)            # [N, in_dim + 3*dense_dim]
        return self.out_proj(cat3)


# ──────────────────────────────────────────────────────────────────────
# 4. GAT Encoder + Residual Dense Head
# ──────────────────────────────────────────────────────────────────────
class GATWithDenseHead(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=8, dense_dim=128, head_drop=0.5):
        super().__init__()

        # ── GAT Encoder ──
        # Layer 1: Multi-head attention
        # Output dim = hidden_channels * heads (e.g. 64 * 8 = 512)
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.bn1 = BatchNorm1d(hidden_channels * heads)

        # Residual skip to match conv1 output dimension
        self.res1 = nn.Linear(in_channels, hidden_channels * heads)

        # Layer 2: concat=False averages heads -> output dim = hidden_channels
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels,
                             heads=heads, concat=False)
        self.bn2 = BatchNorm1d(hidden_channels)

        # Residual skip to match conv2 output dimension
        self.res2 = nn.Linear(hidden_channels * heads, hidden_channels)

        # ── Residual Dense Classification Head ──
        self.head = ResidualDenseHead(
            in_dim=hidden_channels,
            dense_dim=dense_dim,
            out_dim=out_channels,
            drop_p=head_drop,
        )

    def forward(self, x, edge_index):
        # --- Encoder block 1 ---
        identity = self.res1(x)
        x = self.conv1(x, edge_index) + identity
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)

        # --- Encoder block 2 ---
        identity = self.res2(x)
        x = self.conv2(x, edge_index) + identity
        x = self.bn2(x)
        x = F.elu(x)

        # --- Residual Dense Head ---
        return self.head(x)


# ──────────────────────────────────────────────────────────────────────
# 5. Initialization
# ──────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GATWithDenseHead(
    in_channels=dataset.num_node_features,   # 500
    hidden_channels=64,                      # same as original GAT
    out_channels=dataset.num_classes,        # 7
    heads=8,                                 # same as original GAT
    dense_dim=128,                           # width of each dense block
    head_drop=0.5,
).to(device)

data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=10
)


# ──────────────────────────────────────────────────────────────────────
# 6. Training
# ──────────────────────────────────────────────────────────────────────
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)

        loss = F.cross_entropy(
            out[:batch.batch_size],
            batch.y[:batch.batch_size],
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(data.train_mask.sum())


@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


# ──────────────────────────────────────────────────────────────────────
# 7. Execution Loop with Best Model Checkpointing
# ──────────────────────────────────────────────────────────────────────
print(f"Starting GAT + Residual Dense Head Training on {device}...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
best_val_acc = 0.0
best_test_acc = 0.0

for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'GAT_denseHead_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train: {train_acc:.4f}, Val: {val_acc:.4f}, '
              f'Test: {test_acc:.4f}, Best Val: {best_val_acc:.4f}')

print(f"\nFinal Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy at Best Val:      {best_test_acc:.4f}")


Starting GAT + Residual Dense Head Training on cuda...
Model parameters: 889,287
Epoch: 010, Loss: 1.3268, Train: 0.5404, Val: 0.5126, Test: 0.5134, Best Val: 0.5257
Epoch: 020, Loss: 1.2824, Train: 0.5551, Val: 0.5200, Test: 0.5263, Best Val: 0.5280
Epoch: 030, Loss: 1.2771, Train: 0.5632, Val: 0.5274, Test: 0.5308, Best Val: 0.5293
Epoch: 040, Loss: 1.2773, Train: 0.5761, Val: 0.5269, Test: 0.5287, Best Val: 0.5320
Epoch: 050, Loss: 1.1903, Train: 0.5914, Val: 0.5277, Test: 0.5309, Best Val: 0.5320
Epoch: 060, Loss: 1.1615, Train: 0.6144, Val: 0.5157, Test: 0.5271, Best Val: 0.5322
Epoch: 070, Loss: 1.0852, Train: 0.6385, Val: 0.5261, Test: 0.5332, Best Val: 0.5322
Epoch: 080, Loss: 1.0182, Train: 0.6757, Val: 0.5179, Test: 0.5258, Best Val: 0.5322
Epoch: 090, Loss: 0.9796, Train: 0.6959, Val: 0.5175, Test: 0.5222, Best Val: 0.5322
Epoch: 100, Loss: 0.9524, Train: 0.7024, Val: 0.5143, Test: 0.5200, Best Val: 0.5322
Epoch: 110, Loss: 0.9423, Train: 0.7067, Val: 0.5148, Test: 0.5203, B